In [5]:
%matplotlib inline
import os, cv2, numpy as np, matplotlib.pyplot as plt
from ultralytics import YOLO

print("OpenCV:", cv2.__version__)


OpenCV: 4.9.0


In [6]:
VIDEO_PATH = r"C:\Users\islam\SportsPose\data\data\videos\IMG_5346.mp4"
OUT_PATH   = r"C:\Users\islam\SportsPose\results\IMG_5346_skeleton.mp4"

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
print("Input video:", VIDEO_PATH)


Input video: C:\Users\islam\SportsPose\data\data\videos\IMG_5346.mp4


In [7]:
# You need the small pose model; download if missing
model = YOLO("yolov8s-pose.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
fps  = cap.get(cv2.CAP_PROP_FPS)
w    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out  = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

COCO_EDGES = [
    (5,7),(7,9), (6,8),(8,10), (5,6), (11,12), (5,11),(6,12),
    (11,13),(13,15),(12,14),(14,16)
]

frame_i = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    res = model(frame, verbose=False)[0]
    poses = res.keypoints.xy.cpu().numpy() if res.keypoints is not None else []

    for p in poses:
        for x,y in p.astype(int):
            cv2.circle(frame, (x,y), 3, (0,255,0), -1)
        for a,b in COCO_EDGES:
            pa, pb = p[a].astype(int), p[b].astype(int)
            cv2.line(frame, tuple(pa), tuple(pb), (255,0,0), 2)

    out.write(frame)
    frame_i += 1
    if frame_i % 30 == 0:
        print(f"Processed {frame_i} frames...")

cap.release()
out.release()
print(f"✅ Saved skeleton video → {OUT_PATH}")


Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processed 390 frames...
Processed 420 frames...
Processed 450 frames...
Processed 480 frames...
Processed 510 frames...
Processed 540 frames...
Processed 570 frames...
Processed 600 frames...
Processed 630 frames...
Processed 660 frames...
Processed 690 frames...
Processed 720 frames...
Processed 750 frames...
✅ Saved skeleton video → C:\Users\islam\SportsPose\results\IMG_5346_skeleton.mp4


In [8]:
cap = cv2.VideoCapture(OUT_PATH)
nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, nframes // 2)  # middle frame
ret, frame = cap.read()
cap.release()

plt.figure(figsize=(8,5))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title("Sample frame with pose overlay")
plt.show()


<Figure size 800x500 with 1 Axes>

In [9]:
%matplotlib inline
import os, cv2, numpy as np, matplotlib.pyplot as plt
from ultralytics import YOLO

VIDEO_PATH = r"C:\Users\islam\SportsPose\data\data\videos\IMG_5346.mp4"
OUT_2D     = r"C:\Users\islam\SportsPose\results\IMG_5346_skeleton.mp4"      # (from earlier)
OUT_3D     = r"C:\Users\islam\SportsPose\results\IMG_5346_3d.mp4"            # new 3D animation

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
print("Video:", VIDEO_PATH)


Video: C:\Users\islam\SportsPose\data\data\videos\IMG_5346.mp4


In [10]:
# If you haven't installed yet:
# !pip install ultralytics opencv-python matplotlib numpy

model = YOLO("yolov8s-pose.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
fps  = cap.get(cv2.CAP_PROP_FPS)
nF   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

COCO_EDGES = [
    (5,7),(7,9), (6,8),(8,10), (5,6), (11,12), (5,11),(6,12),
    (11,13),(13,15),(12,14),(14,16)
]

poses2d_frames = []  # list of [ (n_person, 17, 2) ] per frame
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    res = model(frame, verbose=False)[0]
    P = res.keypoints.xy.cpu().numpy() if res.keypoints is not None else np.empty((0,17,2))
    poses2d_frames.append(P)
    i += 1
    if i % 30 == 0:
        print(f"2D processed {i}/{nF} frames")
cap.release()

print("Frames collected:", len(poses2d_frames))


2D processed 30/776 frames
2D processed 60/776 frames
2D processed 90/776 frames
2D processed 120/776 frames
2D processed 150/776 frames
2D processed 180/776 frames
2D processed 210/776 frames
2D processed 240/776 frames
2D processed 270/776 frames
2D processed 300/776 frames
2D processed 330/776 frames
2D processed 360/776 frames
2D processed 390/776 frames
2D processed 420/776 frames
2D processed 450/776 frames
2D processed 480/776 frames
2D processed 510/776 frames
2D processed 540/776 frames
2D processed 570/776 frames
2D processed 600/776 frames
2D processed 630/776 frames
2D processed 660/776 frames
2D processed 690/776 frames
2D processed 720/776 frames
2D processed 750/776 frames
Frames collected: 776


In [11]:
def lift_fake(p2d):
    """
    p2d: (17,2) pixels -> (17,3) with a simple depth cue from vertical position.
    This is NOT true metrically-correct 3D; it's for visualization only.
    """
    if p2d.shape[0] != 17:
        return None
    y = p2d[:,1]
    z = -0.002 * (y - y.mean())
    return np.c_[p2d, z]

poses3d_frames = []  # list of [ (n_person, 17,3) ] per frame
for idx, P in enumerate(poses2d_frames):
    if P.size == 0:
        poses3d_frames.append([])
        continue
    P3s = []
    for p in P:
        p3 = lift_fake(p)
        if p3 is not None:
            P3s.append(p3)
    poses3d_frames.append(P3s)
    if (idx+1) % 30 == 0:
        print(f"3D lifted {idx+1}/{len(poses2d_frames)} frames")

print("Ready to render 3D video")


3D lifted 30/776 frames
3D lifted 60/776 frames
3D lifted 90/776 frames
3D lifted 120/776 frames
3D lifted 150/776 frames
3D lifted 180/776 frames
3D lifted 210/776 frames
3D lifted 240/776 frames
3D lifted 270/776 frames
3D lifted 300/776 frames
3D lifted 330/776 frames
3D lifted 360/776 frames
3D lifted 390/776 frames
3D lifted 420/776 frames
3D lifted 450/776 frames
3D lifted 480/776 frames
3D lifted 510/776 frames
3D lifted 540/776 frames
3D lifted 570/776 frames
3D lifted 600/776 frames
3D lifted 630/776 frames
3D lifted 660/776 frames
3D lifted 690/776 frames
3D lifted 720/776 frames
3D lifted 750/776 frames
Ready to render 3D video


In [12]:
import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

# If saving fails due to ffmpeg, install it first:
# conda install -c conda-forge ffmpeg

W, H = 960, 540  # canvas size of the rendered 3D animation
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
vw3d   = cv2.VideoWriter(OUT_3D, fourcc, fps, (W, H))

def draw_frame(P3s, elev=20, azim=-70):
    fig = Figure(figsize=(W/100, H/100), dpi=100)
    canvas = FigureCanvas(fig)
    ax = fig.add_subplot(111, projection="3d")

    # nice autoscale
    all_pts = []
    for p3 in P3s:
        all_pts.append(p3)
    if len(all_pts):
        all_pts = np.concatenate(all_pts, axis=0)
        rng = (all_pts.max(axis=0) - all_pts.min(axis=0))
        s = max(rng[0], rng[1], rng[2], 1e-6)
    else:
        s = 1.0

    # draw each person
    for p3 in P3s:
        for a,b in [(5,7),(7,9),(6,8),(8,10),(5,6),(11,12),(5,11),(6,12),(11,13),(13,15),(12,14),(14,16)]:
            ax.plot([p3[a,0], p3[b,0]], [p3[a,1], p3[b,1]], [p3[a,2], p3[b,2]])
        ax.scatter(p3[:,0], p3[:,1], p3[:,2], s=8)

    ax.set_xlim([-s/2, s/2]); ax.set_ylim([-s/2, s/2]); ax.set_zlim([-s/2, s/2])
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.view_init(elev=elev, azim=azim)
    fig.tight_layout()

    canvas.draw()
    buf = np.frombuffer(canvas.tostring_rgb(), dtype=np.uint8)
    img = buf.reshape(int(fig.bbox.bounds[3]), int(fig.bbox.bounds[2]), 3)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

# render sequence
for idx, P3s in enumerate(poses3d_frames):
    frame_img = draw_frame(P3s, elev=20, azim=-70)
    vw3d.write(frame_img)
    if (idx+1) % 30 == 0:
        print(f"Rendered 3D frame {idx+1}/{len(poses3d_frames)}")

vw3d.release()
print(f"✅ Saved 3D animation → {OUT_3D}")


C:\Users\islam\AppData\Local\Temp\ipykernel_29416\2319601312.py:40: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  buf = np.frombuffer(canvas.tostring_rgb(), dtype=np.uint8)


Rendered 3D frame 30/776
Rendered 3D frame 60/776
Rendered 3D frame 90/776
Rendered 3D frame 120/776
Rendered 3D frame 150/776
Rendered 3D frame 180/776
Rendered 3D frame 210/776
Rendered 3D frame 240/776
Rendered 3D frame 270/776
Rendered 3D frame 300/776
Rendered 3D frame 330/776
Rendered 3D frame 360/776
Rendered 3D frame 390/776
Rendered 3D frame 420/776
Rendered 3D frame 450/776
Rendered 3D frame 480/776
Rendered 3D frame 510/776
Rendered 3D frame 540/776
Rendered 3D frame 570/776
Rendered 3D frame 600/776
Rendered 3D frame 630/776
Rendered 3D frame 660/776
Rendered 3D frame 690/776
Rendered 3D frame 720/776
Rendered 3D frame 750/776
✅ Saved 3D animation → C:\Users\islam\SportsPose\results\IMG_5346_3d.mp4
